# Lab 2: Graph Classification

#### Load packages

In [1]:
# Helper function for visualization.
%matplotlib inline
import networkx as nx
import matplotlib.pyplot as plt

## Load and format the dataset

Load **ENZYMES dataset** provided by the TU Dortmund University that has collected a wide range of different graph classification datasets, known as the [**TUDatasets**](https://chrsmrrs.github.io/datasets/), which are also accessible via [`torch_geometric.datasets.TUDataset`](https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html#torch_geometric.datasets.TUDataset) in PyTorch Geometric.

In [4]:
from torch_geometric.datasets import TUDataset

dataset = TUDataset('./','ENZYMES')
print(f'Dataset: {dataset} is loaded')
print('======================')

Processing...


Dataset: ENZYMES(600) is loaded


Done!


Investigate the dataset

In [11]:
print(dataset.num_classes)

print(dataset.num_features)

data = dataset[0]  # Get the first graph object.

print(data)
print('==============================================================')

# Gather some statistics about the graph.
# TO BE COMPLETED
print("Graph dirigé: " ,data.is_directed())
print("Nombre de noeuds du graph: ",data.num_nodes)
print("Nombre de feature par noeud:" ,data.num_features)
print("Nombre d'edge du graph: ",data.num_edges)
print("Nombre de feature par edges: ",data.num_edge_features)
print("Self loop: ",data.has_self_loops())
print("Noeud isolé dans le graph: ",data.has_isolated_nodes())

6
3
Data(edge_index=[2, 168], x=[37, 3], y=[1])
Graph dirigé:  False
Nombre de noeuds du graph:  37
Nombre de feature par noeud: 3
Nombre d'edge du graph:  168
Nombre de feature par edges:  0
Self loop:  False
Noeud isolé dans le graph:  False


Split the dataset into training and test sets

You may use `train_test_split` method from `sklearn`

For other datasets, argument `split` can be available: https://pytorch-geometric.readthedocs.io/en/2.8.0.post1/tutorial/dataset_splitting.html

In [12]:
# TO BE COMPLETED
from sklearn.model_selection import train_test_split

train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, shuffle=True, stratify=dataset.data.y.numpy())

/tmp/cache-aikahloul/ipykernel_521049/2825104783.py:4: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, shuffle=True, stratify=dataset.data.y.numpy())


Set the mini-batching [DataLoader object in PyG]([DataLoader](https://pytorch-geometric.readthedocs.io/en/latest/modules/loader.html))

Check: https://pytorch-geometric.readthedocs.io/en/latest/notes/introduction.html?highlight=batch#mini-batches

In [13]:
# TO BE COMPLETED
from torch_geometric.loader import DataLoader
train_loader= DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader= DataLoader(test_dataset, batch_size=64, shuffle=True)

## Implementing a Graph Neural Network for graph classification task

Define the architecture of a four layers GNN (3 hidden layers GCN + 1 fully-connected layer, don't forget readout operation):

In [14]:
import torch
from torch.nn import Linear
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self,num_features=dataset.num_features, num_classes=dataset.num_classes):
        super().__init__()
        torch.manual_seed(1234)
        self.conv1 = GCNConv(num_features, 4)
        self.conv2 = GCNConv(4, 4)
        self.conv3 = GCNConv(4, 2)
        self.classifier = Linear(2, num_classes)

    def forward(self, x, edge_index,batch):
        h = self.conv1(x, edge_index)
        h = h.tanh()
        h = self.conv2(h, edge_index)
        h = h.tanh()
        h = self.conv3(h, edge_index)
        h = h.tanh()  # Final GNN embedding space.
        h = global_mean_pool(x, batch)
        h = self.lin(h)
        
        return h

model = GCN()
print(model)

GCN(
  (conv1): GCNConv(3, 4)
  (conv2): GCNConv(4, 4)
  (conv3): GCNConv(4, 2)
  (classifier): Linear(in_features=2, out_features=6, bias=True)
)


Define your training and evaluation loops

You can reuse the one from your previous TP

In [16]:
# TO BE COMPLETED
criterion = torch.nn.CrossEntropyLoss()  
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train(model, data, optimizer, criterion, epochs=200, verbose=True):

    for epoch in range(epochs + 1):
        
        model.train()  # Mode entraînement
        optimizer.zero_grad()  # Reset des gradients
        out,_ = model(data.x, data.edge_index)  # Forward
        loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Perte
        loss.backward()
        optimizer.step()
        
        # Évaluation sur le jeu d'entraînement et de test
        model.eval()
        with torch.no_grad():
            out,_ = model(data.x, data.edge_index)
            pred = F.softmax(out).argmax(dim=1)
            acc_train = (pred[data.train_mask] == data.y[data.train_mask]).sum() / data.train_mask.sum()
            acc_test = (pred[~data.train_mask] == data.y[~data.train_mask]).sum() / (~data.train_mask).sum()

        if verbose and epoch % 20 == 0:
            print(f"Époque {epoch:03d} | Loss: {loss:.4f} | Accuracy train: {acc_train:.4f} | Accuracy test: {acc_test:.4f}")

    return model

train(model,data,optimizer,criterion)

TypeError: GCN.forward() missing 1 required positional argument: 'batch'